# 04 · Uplift Modeling

**Data:** `uci_uplift_campaigns.parquet` (synthetic RCT on UCI customers)  
**Learners:** T-learner, X-learner, Uplift Random Forest (class transformation)

**Output:** incremental impact of discount offer (CATE).

Split: 60/20/20 by customer (both campaign waves in same split group).


In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def find_project_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in (path, *path.parents):
        if (candidate / "scripts" / "build_datasets.py").exists():
            return candidate
    return path


PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "data" / "modeling"
MODELS = PROJECT_ROOT / "models"
MODELS.mkdir(exist_ok=True)


def load_parquet(name: str) -> pd.DataFrame:
    path = DATA / name
    if not path.exists():
        raise FileNotFoundError(f"Missing {path} — run: python scripts/uci_pipeline.py")
    return pd.read_parquet(path)


def save_artifact(name: str, obj) -> Path:
    path = MODELS / name
    joblib.dump(obj, path)
    print(f"Saved → {path.relative_to(PROJECT_ROOT)}")
    return path


def audit_and_clean(
    df: pd.DataFrame,
    *,
    subset: list[str] | None = None,
    id_col: str | None = None,
    required_cols: list[str] | None = None,
    label: str = "dataset",
) -> pd.DataFrame:
    """Report and drop duplicate rows + rows with NA in required columns (before split)."""
    out = df.copy()
    n0 = len(out)
    dup_subset = subset if subset is not None else ([id_col] if id_col else None)
    n_dup = out.duplicated(subset=dup_subset, keep="first").sum() if dup_subset else out.duplicated(keep="first").sum()
    if n_dup:
        out = out.drop_duplicates(subset=dup_subset, keep="first")
    req = [c for c in (required_cols or []) if c in out.columns]
    na_rows = out[req].isna().any(axis=1).sum() if req else 0
    na_by_col = out[req].isna().sum()
    if req:
        out = out.dropna(subset=req)
    print(
        f"[{label}] {n0:,} rows -> {len(out):,} | "
        f"dropped {n_dup:,} duplicates, {na_rows:,} rows with NA"
    )
    if na_rows and (na_by_col > 0).any():
        print("  NA counts:", na_by_col[na_by_col > 0].to_dict())
    return out


In [ ]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import xgboost as xgb

df = load_parquet("uci_uplift_campaigns.parquet")
FEATURES = ["discount_sensitivity", "engagement_score", "avg_order_value"]
df = audit_and_clean(
    df,
    subset=["customer_id", "campaign_id"],
    required_cols=["customer_id", "campaign_id", "treatment", "responded", *FEATURES],
    label="uplift_campaigns",
)
X = df[FEATURES]
treatment = df["treatment"].astype(int)
y = df["responded"].astype(int)

print(f"Rows: {len(df):,} · Treatment rate: {treatment.mean():.1%} · Response rate: {y.mean():.1%}")


In [ ]:
corr = X.assign(treatment=treatment, responded=y).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Uplift feature correlation")
plt.tight_layout()
plt.show()


In [ ]:
# Split by customer so both waves stay together
cust = df["customer_id"].astype(str).unique().to_numpy()
c_train, c_temp = train_test_split(cust, test_size=0.40, random_state=RANDOM_STATE)
c_val, c_test = train_test_split(c_temp, test_size=0.50, random_state=RANDOM_STATE)

def mask(ids): return df["customer_id"].isin(ids)
train, val, test = mask(c_train), mask(c_val), mask(c_test)
for name, split in [("train", train), ("val", val), ("test", test)]:
    print(f"{name}: {split.sum():,} rows")


In [ ]:
def uplift_at_k(y_true, uplift, treatment, k=0.3):
    order = np.argsort(uplift)[::-1]
    n = max(1, int(len(y_true) * k))
    top = order[:n]
    tr = treatment[top] == 1
    ct = ~tr
    if tr.sum() == 0 or ct.sum() == 0:
        return 0.0
    return y_true[top][tr].mean() - y_true[top][ct].mean()


def qini_auc_score(y_true, uplift, treatment):
    order = np.argsort(uplift)[::-1]
    y, w = y_true[order], treatment[order]
    cum_tr = np.cumsum(y * w)
    cum_ct = np.cumsum(y * (1 - w))
    cum_n_tr = np.cumsum(w)
    cum_n_ct = np.cumsum(1 - w)
    with np.errstate(divide="ignore", invalid="ignore"):
        qini = cum_tr - cum_n_tr * np.divide(cum_ct, cum_n_ct, out=np.zeros_like(cum_ct, dtype=float), where=cum_n_ct > 0)
    if len(qini) < 2:
        return 0.0
    return float(np.trapezoid(qini / len(qini)) - np.trapezoid(cum_n_tr / len(cum_n_tr) * (y[w == 1].mean() - y[w == 0].mean())))


class TLearner:
    def __init__(self, estimator):
        self.estimator = estimator

    def fit(self, X, y, w):
        X, y, w = np.asarray(X), np.asarray(y), np.asarray(w)
        self.m1_ = clone(self.estimator)
        self.m0_ = clone(self.estimator)
        self.m1_.fit(X[w == 1], y[w == 1])
        self.m0_.fit(X[w == 0], y[w == 0])
        return self

    def predict(self, X):
        X = np.asarray(X)
        p1 = self.m1_.predict_proba(X)[:, 1]
        p0 = self.m0_.predict_proba(X)[:, 1]
        return p1 - p0


class XLearner:
    def __init__(self, estimator):
        self.estimator = estimator

    def fit(self, X, y, w):
        X, y, w = np.asarray(X), np.asarray(y), np.asarray(w)
        t = TLearner(self.estimator)
        t.fit(X, y, w)
        mu1 = t.m1_.predict_proba(X)[:, 1]
        mu0 = t.m0_.predict_proba(X)[:, 1]
        d1 = y[w == 1] - mu0[w == 1]
        d0 = mu1[w == 0] - y[w == 0]
        from sklearn.ensemble import GradientBoostingRegressor
        self.tau1_ = GradientBoostingRegressor(random_state=RANDOM_STATE)
        self.tau0_ = GradientBoostingRegressor(random_state=RANDOM_STATE)
        self.tau1_.fit(X[w == 1], d1)
        self.tau0_.fit(X[w == 0], d0)
        self.p_ = w.mean()
        return self

    def predict(self, X):
        X = np.asarray(X)
        return self.p_ * self.tau0_.predict(X) + (1 - self.p_) * self.tau1_.predict(X)


class UpliftRandomForest:
    def __init__(self, **kwargs):
        self.kwargs = kwargs

    def fit(self, X, y, w):
        X, y, w = np.asarray(X), np.asarray(y), np.asarray(w)
        z = np.where(w == 1, y, 1 - y)
        self.model_ = RandomForestClassifier(**self.kwargs)
        self.model_.fit(X, z)
        return self

    def predict(self, X):
        X = np.asarray(X)
        return 2 * self.model_.predict_proba(X)[:, 1] - 1


base_clf = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    random_state=RANDOM_STATE, verbosity=0,
)

learners = {
    "T-learner": TLearner(base_clf),
    "X-learner": XLearner(base_clf),
    "UpliftRF": UpliftRandomForest(n_estimators=100, max_depth=5, random_state=RANDOM_STATE, n_jobs=-1),
}
results = []

X_train = X.loc[train].values
y_train = y.loc[train].values
w_train = treatment.loc[train].values
X_test = X.loc[test].values
y_test = y.loc[test].values
w_test = treatment.loc[test].values

for name, model in learners.items():
    model.fit(X_train, y_train, w_train)
    uplift_scores = model.predict(X_test)
    qini = qini_auc_score(y_test, uplift_scores, w_test)
    uplift30 = uplift_at_k(y_test, uplift_scores, w_test, k=0.3)
    results.append({"model": name, "qini_auc": qini, "uplift_at_30pct": uplift30, "estimator": model})

leaderboard = pd.DataFrame([{k: v for k, v in r.items() if k != "estimator"} for r in results]).set_index("model")
display(leaderboard.round(4))
best = max(results, key=lambda r: r["qini_auc"])
print(f"Best: {best['model']}")


In [ ]:
save_artifact("04_uplift_best.joblib", {
    "model_name": best["model"],
    "model": best["estimator"],
    "features": FEATURES,
    "metrics": {k: best[k] for k in ("qini_auc", "uplift_at_30pct")},
})
